In [1]:
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd


keyword = "데이터분석"
base_url = "https://www.saramin.co.kr"
search_url = (
    f"{base_url}/zf_user/search"
    f"?search_area=main&search_done=y&search_optional_item=n"
    f"&searchType=search&searchword={keyword}"
)

headers = {
    "User-Agent": "Mozilla/5.0"
}


res = requests.get(search_url, headers=headers)
res.raise_for_status()

soup = BeautifulSoup(res.text, "lxml")


items = soup.select("div.item_recruit")

print("공고 개수:", len(items))

rows = []

for item in items:

    company_tag = item.select_one(".area_corp a, .area_corp strong, .corp_name a")
    col_company = company_tag.get_text(strip=True) if company_tag else ""


    title_tag = item.select_one("h2.job_tit a, a.str_tit")
    col_recruit = title_tag.get_text(strip=True) if title_tag else ""


    cond_spans = item.select(".job_condition span")
    cond_texts = [span.get_text(strip=True) for span in cond_spans if span.get_text(strip=True)]
    col_detail = " | ".join(cond_texts)


    col_url = ""
    if title_tag and title_tag.has_attr("href"):
        href = title_tag["href"]
        if not href.startswith("http"):
            href = base_url + href
        col_url = href


    if not col_recruit:
        continue

    rows.append(
        {
            "Site": "saramin",
            "Col_Company": col_company,
            "Col_Recruit": col_recruit,
            "Col_detail": col_detail,
            "Col_url": col_url,
        }
    )

df = pd.DataFrame(rows)


os.makedirs("data_tmp", exist_ok=True)
output_path = os.path.join("data_tmp", "data_saramin.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

output_path, df.head()


공고 개수: 40


('data_tmp\\data_saramin.csv',
       Site Col_Company                    Col_Recruit  \
 0  saramin      (주)베가스  [Begas] 2026데이터분석전문가 신입·경력 모집   
 1  saramin      (주)이엠넷               데이터분석가 부문 경력직 채용   
 2  saramin   아이비커리어(주)        [통근버스]삼성물산 본사 IT데이터분석채용   
 3  saramin   넛지헬스케어(주)         [캐시워크]데이터분석담당 채용전환형 인턴   
 4  saramin   (주)나눔스페이스           (주)나눔스페이스데이터분석팀 채용공고   
 
                                     Col_detail  \
 0                     서울중구 | 신입·경력 | 석사↑ | 정규직   
 1                    서울구로구 | 경력1년↑ | 대졸↑ | 정규직   
 2                서울강동구 | 경력 3~20년 | 학력무관 | 파견직   
 3                       서울강남구 | 신입 | 대졸↑ | 인턴직   
 4  전북전주시 완산구 | 경력무관 | 대졸↑ | 기간제·계약직 | 3,240 만원   
 
                                              Col_url  
 0  https://www.saramin.co.kr/zf_user/jobs/relay/v...  
 1  https://www.saramin.co.kr/zf_user/jobs/relay/v...  
 2  https://www.saramin.co.kr/zf_user/jobs/relay/v...  
 3  https://www.saramin.co.kr/zf_user/jobs/relay/v...  
 4  https://www.saramin.co.kr/z